# 03 EAP Name Resampling

Replicate the EAP demo setting. EAP is trained with `train_size=100` in two conditions: `normal_order_42` uses the original IOI training split and `resampled_order_43` uses the same split after training-wise IOI name resampling. Both conditions are evaluated on the same untransformed test split. The top-k sweep and finalization match the notebook configuration.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from circuit_discovery.circuit import overlap_stats
from circuit_discovery.run import (
    get_compute_device,
    load_configs,
    load_model,
    load_eap_resampled_conditions_from_config,
    train_loader_from_config,
    eval_loader_from_config,
    top_k_values_from_config,
    evaluate_circuit,
)
from circuit_discovery.visualization import load_circuit

configs = load_configs()
params = configs["notebooks"]["03_eap_name_sensitivity"]["hyperparams"]
eap_cfg = configs["artifacts"]["eap"]
print("project root:", PROJECT_ROOT)
print("device:", get_compute_device())
params


In [ ]:
# Optional regeneration. Expensive; leave disabled when browsing saved artifacts.
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    from circuit_discovery.algorithms.eap import EAP, EAPConfig
    from circuit_discovery.metrics import discogp_fidelity_loss

    output_dir = PROJECT_ROOT / eap_cfg["root"]
    output_dir.mkdir(parents=True, exist_ok=True)
    conditions = load_eap_resampled_conditions_from_config(params)
    ks = top_k_values_from_config(params)

    for condition in eap_cfg["conditions"]:
        model = load_model(params["model_name"])
        runner = EAP(
            model=model,
            config=EAPConfig(
                model_name=params["model_name"],
                absolute_scores=params["absolute_scores"],
                seed=params["seed"],
                tqdm_disabled=False,
            ),
        )
        result = runner.fit(
            train_loader_from_config(
                conditions[condition].train,
                params,
                seed=conditions[condition].train_order_seed,
            ),
            loss_fn=discogp_fidelity_loss,
        )
        for k in ks:
            circuit = result.circuit_for_top_k(k, model=model, finalize=True)
            rel_path = eap_cfg["path_template"].format(condition=condition, top_k=k)
            torch.save({"circuit": circuit, "algorithm": "eap", "condition": condition, "top_k": k}, PROJECT_ROOT / rel_path)
        print("saved", condition, "top-k circuits")


In [ ]:
conditions = load_eap_resampled_conditions_from_config(params)
selected_ks = eap_cfg["selected_top_k"]
rows = []

for k in selected_ks:
    loaded = {}
    evals = {}
    for condition in eap_cfg["conditions"]:
        path = PROJECT_ROOT / eap_cfg["path_template"].format(condition=condition, top_k=k)
        print(condition, "k=", k, "->", path, "exists=", path.exists())
        if path.exists():
            circuit = load_circuit(path)
            loaded[condition] = circuit
            evals[condition] = evaluate_circuit(
                load_model(params["model_name"]),
                eval_loader_from_config(conditions[condition].test, params),
                circuit,
            )
    if set(loaded) == set(eap_cfg["conditions"]):
        left, right = eap_cfg["conditions"]
        stats = overlap_stats(loaded[left], loaded[right])
        rows.append({
            "top_k": k,
            f"{left}_acc": evals[left].get("acc"),
            f"{right}_acc": evals[right].get("acc"),
            f"{left}_edge_density": evals[left].get("edge_density"),
            f"{right}_edge_density": evals[right].get("edge_density"),
            "edge_iou": stats.get("edge_jaccard"),
            "node_iou": stats.get("node_jaccard"),
        })

display(rows)
